In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import bisect
from google.colab import drive

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# --- Black-Scholes 核心函數 ---
def bs_price(S, K, T, r, sigma, option_type='C'):
    if sigma <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'C':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def find_iv(market_price, S, K, T, r, option_type='C'):
    intrinsic = max(0, S - K) if option_type == 'C' else max(0, K - S)
    effective_market_price = max(market_price, intrinsic + 0.05)
    target_func = lambda sigma: bs_price(S, K, T, r, sigma, option_type) - effective_market_price
    try:
        # 搜尋 IV 區間
        return bisect(target_func, 0.0001, 5.0, xtol=1e-4)
    except:
        return np.nan

# --- 設定檔案路徑 ---
index_path = '/content/drive/My Drive/Index_411042033_2024.xlsx - Sheet1.csv'
folder_path = '/content/drive/My Drive/金融資料探勘/Option_2024'
output_file = '/content/drive/My Drive/金融資料探勘/Final_Analysis_Report_2024_Success.csv'

# 讀取索引參數
index_df = pd.read_csv(index_path)
index_df.columns = [str(c).strip() for c in index_df.columns]

final_results = []

print("🚀 開始進行深度資料融合分析...")

for _, row in index_df.iterrows():
    target_file = str(row['File']).strip()
    S0 = float(row['S0'])
    Rf = float(row['Rf'])
    T_years = float(row['Maturity']) / 365

    full_path = os.path.join(folder_path, target_file)

    if not os.path.exists(full_path):
        print(f"⚠️ 找不到檔案: {target_file}")
        continue

    # 讀取每日交易資料（自動偵測編碼）
    try:
        df_raw = pd.read_csv(full_path, encoding='big5', skiprows=[1])
    except:
        df_raw = pd.read_csv(full_path, encoding='utf-8', skiprows=[1])

    # 徹底清洗欄位名稱：移除空格、換行、轉字串
    df_raw.columns = [str(c).strip() for c in df_raw.columns]

    # 強制轉換關鍵欄位為數字
    for col in ['成交價格', '履約價格', '成交數量(B or S)']:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

    # 篩選條件：成交量 >= 20 且不能有 NaN
    df_day = df_raw[df_raw['成交數量(B or S)'] >= 20].dropna(subset=['成交價格', '履約價格']).copy()

    if df_day.empty:
        print(f"ℹ️ {target_file} 在篩選條件下無資料")
        continue

    # --- 精確計算分組指標 ---
    # 這裡使用 .str.contains 避免標籤裡有隱藏空格 (例如 'C ' 或 ' C')
    calls = df_day[df_day['買賣權別'].str.contains('C', na=False)]
    puts = df_day[df_day['買賣權別'].str.contains('P', na=False)]

    # 二分法反推 IV
    c_ivs = [find_iv(r['成交價格'], S0, r['履約價格'], T_years, Rf, 'C') for _, r in calls.iterrows()]
    p_ivs = [find_iv(r['成交價格'], S0, r['履約價格'], T_years, Rf, 'P') for _, r in puts.iterrows()]

    c_ivs = [x for x in c_ivs if not np.isnan(x)]
    p_ivs = [x for x in p_ivs if not np.isnan(x)]

    # 計算 PCR (Put 總成交量 / Call 總成交量)
    total_call_vol = calls['成交數量(B or S)'].sum()
    total_put_vol = puts['成交數量(B or S)'].sum()
    pcr = total_put_vol / total_call_vol if total_call_vol > 0 else 0

    # 整理結果
    final_results.append({
        'Date': row['Date'],
        'Total_Vol_Call': total_call_vol,
        'Num_Call': len(calls),
        'Mean_IV_Call': np.mean(c_ivs) if c_ivs else 0,
        'Std_IV_Call': np.std(c_ivs, ddof=1) if len(c_ivs) > 1 else 0,
        'Total_Vol_Put': total_put_vol,
        'Num_Put': len(puts),
        'Mean_IV_Put': np.mean(p_ivs) if p_ivs else 0,
        'Std_IV_Put': np.std(p_ivs, ddof=1) if len(p_ivs) > 1 else 0,
        'PCR_Volume': pcr
    })
    print(f"✅ 成功計算: {row['Date']} | S0: {S0} | 有效 IV 筆數: {len(c_ivs) + len(p_ivs)}")

# 儲存
if final_results:
    output_df = pd.DataFrame(final_results)
    output_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n✨ 全部完成！報表已存至: {output_file}")
else:
    print("\n❌ 錯誤：完全沒有產生任何數據，請檢查資料夾中的 CSV 是否為空的。")